Modele de clasificare pentru datasetul Electricity

In [1]:
from google.colab import files
uploaded = files.upload()

Saving electricity-normalized.arff to electricity-normalized.arff


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import arff

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import json

In [3]:
import pandas as pd
from scipy.io import arff

data, meta = arff.loadarff('electricity-normalized.arff')
df = pd.DataFrame(data)

for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, (bytes, bytearray)) else x)

df.head()
df = pd.DataFrame(data)

# convertim bytes -> string (ARFF face asta uneori)
for col in df.columns:
    if df[col].dtype == object:
        df[col] = df[col].apply(lambda x: x.decode('utf-8') if isinstance(x, (bytes, bytearray)) else x)

df.head()

,date,day,period,nswprice,nswdemand,vicprice,vicdemand,transfer,class
0,0.0,2,0.000000,0.056443,0.439155,0.003467,0.422915,0.414912,UP
1,0.0,2,0.021277,0.051699,0.415055,0.003467,0.422915,0.414912,UP
2,0.0,2,0.042553,0.051489,0.385004,0.003467,0.422915,0.414912,UP
3,0.0,2,0.063830,0.045485,0.314639,0.003467,0.422915,0.414912,UP
4,0.0,2,0.085106,0.042482,0.251116,0.003467,0.422915,0.414912,DOWN


In [4]:
X = df.drop(columns=['class'])
y = df['class']

# y e UP/DOWN (string), il facem 0/1
le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((36249, 8), (9063, 8))

1. Logistic Regression

In [5]:
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000))
])

params_lr = {
    'clf__C': [0.1, 1, 10],
    'clf__penalty': ['l2'],
    'clf__solver': ['lbfgs']
}

gs_lr = GridSearchCV(pipe_lr, params_lr, cv=5, scoring='f1', n_jobs=-1)
gs_lr.fit(X_train, y_train)

gs_lr.best_params_, gs_lr.best_score_

({'clf__C': 10, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'},
 np.float64(0.6763224691221792))

2. Decision Tree
``

In [6]:
dt = DecisionTreeClassifier(random_state=42)

params_dt = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 10, 20],
    'min_samples_leaf': [1, 5, 10]
}

gs_dt = GridSearchCV(dt, params_dt, cv=5, scoring='f1', n_jobs=-1)
gs_dt.fit(X_train, y_train)

gs_dt.best_params_, gs_dt.best_score_


({'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2},
 np.float64(0.8536577675649918))

3. Random Forest

In [9]:
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

params_rf = {
    'n_estimators': [100, 150, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5]
}

rs_rf = RandomizedSearchCV(
    rf,
    params_rf,
    n_iter=10,        # doar 10 combinații (rapid)
    cv=3,
    scoring='f1',
    n_jobs=-1,
    random_state=42
)

rs_rf.fit(X_train, y_train)

print(rs_rf.best_params_)
print(rs_rf.best_score_)

best_rf = rs_rf.best_estimator_

{'n_estimators': 150, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': None}
0.8579260467943052


Compararea modelelor pe setul de test

In [10]:
def eval_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)

    # roc_auc doar daca avem predict_proba
    auc = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, proba)

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc
    }

best_lr = gs_lr.best_estimator_
best_dt = gs_dt.best_estimator_
best_rf = rs_rf.best_estimator_

results = []
results.append(eval_model("LogisticRegression", best_lr, X_test, y_test))
results.append(eval_model("DecisionTree", best_dt, X_test, y_test))
results.append(eval_model("RandomForest", best_rf, X_test, y_test))

results_df = pd.DataFrame(results)
results_df

,model,accuracy,precision,recall,f1,roc_auc
0,LogisticRegression,0.756703,0.774841,0.601871,0.677490,0.821018
1,DecisionTree,0.887234,0.868545,0.865385,0.866962,0.884370
2,RandomForest,0.894406,0.890148,0.857069,0.873295,0.963004


Salvarea rezultatelor brute


In [13]:
results_df.to_csv("results_metrics.csv", index=False)

best_params = {
    "LogisticRegression": gs_lr.best_params_,
    "DecisionTree": gs_dt.best_params_,
    "RandomForest": rs_rf.best_params_
}

with open("best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("Salvat: results_metrics.csv si best_params.json")

Salvat: results_metrics.csv si best_params.json


Din rezultatele obtinute se observa ca Random Forest are performanta cea mai buna.
Decision Tree functioneaza destul de bine, iar Logistic Regression are rezultate mai slabe.


In [14]:
importances = best_rf.feature_importances_

for col, imp in zip(X.columns, importances):
    print(col, round(imp, 3))

date 0.177
day 0.052
period 0.102
nswprice 0.342
nswdemand 0.113
vicprice 0.11
vicdemand 0.057
transfer 0.047


S-a analizat importanta caracteristicilor folosind Random Forest.

Se observa ca variabilele legate de cerere si pret (nswdemand, nswprice)
au cea mai mare influenta asupra modelului.

Modelul confirma ceea ce este de asteptat in realitate,
si anume ca cererea influenteaza direct pretul energiei.

Modelul poate face erori in cazul valorilor apropiate,
unde diferenta dintre clase nu este clara.
